In [0]:
# ==========================================
# GOLD LAYER - DRIVER ANALYTICS
# ==========================================

# Read cleaned and enriched data from Silver layer
silver_df = spark.table("silver_vehicle_tracking")

# Validate Silver data loaded successfully
silver_df.select(
    "VehicleId", "VehicleNo", "drivername", "DateTime",
    "speed", "distance_km", "overspeed_flag",
    "harsh_brake_flag", "harsh_acc_flag", "idle_flag"
).show(5)

# Check total Silver record count
silver_df.count()

+---------+----------+----------+-------------------+-----+--------------------+--------------+----------------+--------------+---------+
|VehicleId| VehicleNo|drivername|           DateTime|speed|         distance_km|overspeed_flag|harsh_brake_flag|harsh_acc_flag|idle_flag|
+---------+----------+----------+-------------------+-----+--------------------+--------------+----------------+--------------+---------+
|      185|TS07UK3537|  driver33|2026-04-01 09:44:49|  5.0|                 0.0|             0|               0|             0|        0|
|      185|TS07UK3537|  driver33|2026-04-01 09:44:50|  9.0|0.002073126000713...|             0|               0|             0|        0|
|      185|TS07UK3537|  driver33|2026-04-01 09:44:51|  9.0|0.002663731657713...|             0|               0|             0|        0|
|      185|TS07UK3537|  driver33|2026-04-01 09:45:07| 10.0| 0.07039153545214658|             0|               0|             0|        0|
|      185|TS07UK3537|  driver33|2

489850

In [0]:
# ==========================================
# GOLD STEP 2 - DRIVER TRIP SUMMARY
# ==========================================

from pyspark.sql.functions import col, sum, avg, max, count, to_date, round

# Create daily trip summary per driver and vehicle
driver_trip_summary = silver_df.withColumn("trip_date", to_date(col("DateTime"))) \
    .groupBy("trip_date", "drivername", "VehicleNo") \
    .agg(
        round(sum("distance_km"), 2).alias("total_distance_km"),
        round(avg("speed"), 2).alias("avg_speed"),
        round(max("speed"), 2).alias("max_speed"),
        sum("idle_flag").alias("idle_records"),
        sum("overspeed_flag").alias("overspeed_events"),
        sum("harsh_brake_flag").alias("harsh_brake_events"),
        sum("harsh_acc_flag").alias("harsh_acc_events"),
        count("*").alias("total_gps_records")
    )

# Save as Gold Delta table
driver_trip_summary.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_driver_trip_summary")


# Validate output
driver_trip_summary.show(10)

+----------+----------+----------+-----------------+---------+---------+------------+----------------+------------------+----------------+-----------------+
| trip_date|drivername| VehicleNo|total_distance_km|avg_speed|max_speed|idle_records|overspeed_events|harsh_brake_events|harsh_acc_events|total_gps_records|
+----------+----------+----------+-----------------+---------+---------+------------+----------------+------------------+----------------+-----------------+
|2026-04-06|  driver24|TS36TA4059|            81.88|     14.7|     61.0|          34|               6|                 7|              10|              486|
|2026-04-10|  driver24|TS36TA4059|           268.11|    12.38|     47.0|         238|               0|                28|              19|             2654|
|2026-04-11|  driver16|TS08UJ6789|            69.73|     6.14|     34.0|         294|               0|                 0|               1|              732|
|2026-04-02|  driver20|TS04UC8236|           129.05|    12

In [0]:
# ==========================================
# GOLD STEP 3 - DRIVER BEHAVIOUR EVENTS
# ==========================================

from pyspark.sql.functions import lit

# Create overspeed events
overspeed_events = silver_df.filter(col("overspeed_flag") == 1) \
    .select(
        "drivername", "VehicleNo", "DateTime", "Latitude", "Longitude",
        "LocationName", "speed"
    ) \
    .withColumn("event_type", lit("Overspeed"))

# Create harsh braking events
harsh_brake_events = silver_df.filter(col("harsh_brake_flag") == 1) \
    .select(
        "drivername", "VehicleNo", "DateTime", "Latitude", "Longitude",
        "LocationName", "speed"
    ) \
    .withColumn("event_type", lit("Harsh Braking"))

# Create harsh acceleration events
harsh_acc_events = silver_df.filter(col("harsh_acc_flag") == 1) \
    .select(
        "drivername", "VehicleNo", "DateTime", "Latitude", "Longitude",
        "LocationName", "speed"
    ) \
    .withColumn("event_type", lit("Harsh Acceleration"))

# Create idle events
idle_events = silver_df.filter(col("idle_flag") == 1) \
    .select(
        "drivername", "VehicleNo", "DateTime", "Latitude", "Longitude",
        "LocationName", "speed"
    ) \
    .withColumn("event_type", lit("Idle"))

# Combine all behaviour events into one Gold table
driver_behaviour_events = overspeed_events \
    .unionByName(harsh_brake_events) \
    .unionByName(harsh_acc_events) \
    .unionByName(idle_events)

# Save Gold behaviour events table
driver_behaviour_events.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_driver_behaviour_events")

# Validate event output
driver_behaviour_events.groupBy("event_type").count().show()

+------------------+------+
|        event_type| count|
+------------------+------+
|         Overspeed| 16013|
|     Harsh Braking|  6538|
|Harsh Acceleration|  5853|
|              Idle|144729|
+------------------+------+



In [0]:
# ==========================================
# GOLD STEP 4 - DRIVER SAFETY SCORE
# ==========================================

# Import required functions
from pyspark.sql.functions import col, lit, greatest

# Create daily safety score per driver
# Logic:
# Start with a base score of 100
# Deduct weighted penalties based on risky driving events
# Use 'greatest(0, score)' to ensure score does not go below 0

driver_safety_score = driver_trip_summary.withColumn(
    "safety_score",
    greatest(
        lit(0),  # Ensure minimum score is 0
        lit(100)
        - (col("overspeed_events") * 0.5)   # Mild penalty for overspeeding
        - (col("harsh_brake_events") * 1)   # Higher penalty for harsh braking
        - (col("harsh_acc_events") * 0.8)   # Moderate penalty for harsh acceleration
        - (col("idle_records") * 0.1)       # Small penalty for excessive idling
    )
)

# Save the result as a Delta table (Gold Layer)
# overwrite → replaces existing table (used during development)
# overwriteSchema → allows schema updates (avoids merge conflicts)

driver_safety_score.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_driver_safety_score")

# Validate output by displaying sample records
driver_safety_score.select(
    "trip_date", "drivername", "VehicleNo",
    "overspeed_events", "harsh_brake_events",
    "harsh_acc_events", "idle_records",
    "safety_score"
).show(10)

+----------+----------+----------+----------------+------------------+----------------+------------+------------------+
| trip_date|drivername| VehicleNo|overspeed_events|harsh_brake_events|harsh_acc_events|idle_records|      safety_score|
+----------+----------+----------+----------------+------------------+----------------+------------+------------------+
|2026-04-06|  driver24|TS36TA4059|               6|                 7|              10|          34|              78.6|
|2026-04-10|  driver24|TS36TA4059|               0|                28|              19|         238|              33.0|
|2026-04-11|  driver16|TS08UJ6789|               0|                 0|               1|         294|              69.8|
|2026-04-02|  driver20|TS04UC8236|               1|                22|              22|          95|              50.4|
|2026-04-08|  driver16|TS08UJ6789|               0|                 4|               5|         346|              57.4|
|2026-04-12|  driver10|TS29TA1025|      

In [0]:
# ==========================================
# GOLD STEP 5 - DRIVER RISK RATING
# ==========================================

# Import required function
from pyspark.sql.functions import col, when

# Assign risk category based on safety score
# Logic:
# Low Risk    → score >= 80
# Medium Risk → score between 50 and 79
# High Risk   → score < 50

driver_risk_rating = driver_safety_score.withColumn(
    "risk_rating",
    when(col("safety_score") >= 80, "Low")
    .when((col("safety_score") >= 50) & (col("safety_score") < 80), "Medium")
    .otherwise("High")
)

# Save the result as a Delta table (Gold Layer)
# overwrite → replaces existing table during development
# overwriteSchema → handles schema changes safely

driver_risk_rating.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_driver_risk_rating")

# Validate distribution of drivers across risk categories
driver_risk_rating.groupBy("risk_rating").count().show()

# Display sample records for verification
driver_risk_rating.select(
    "trip_date", "drivername", "VehicleNo",
    "safety_score", "risk_rating"
).show(32)

+-----------+-----+
|risk_rating|count|
+-----------+-----+
|       High|  207|
|        Low|  135|
|     Medium|   87|
+-----------+-----+

+----------+----------+----------+------------------+-----------+
| trip_date|drivername| VehicleNo|      safety_score|risk_rating|
+----------+----------+----------+------------------+-----------+
|2026-04-06|  driver24|TS36TA4059|              78.6|     Medium|
|2026-04-10|  driver24|TS36TA4059|              33.0|       High|
|2026-04-11|  driver16|TS08UJ6789|              69.8|     Medium|
|2026-04-02|  driver20|TS04UC8236|              50.4|     Medium|
|2026-04-08|  driver16|TS08UJ6789|              57.4|     Medium|
|2026-04-12|  driver10|TS29TA1025|              97.5|        Low|
|2026-04-02|  driver25|TS36TA4779|14.399999999999999|       High|
|2026-04-10|  driver15| TG08V7754|               0.0|       High|
|2026-04-05|  driver16|TS08UJ6789|              62.8|     Medium|
|2026-04-06|  driver15| TG08V7754|               0.0|       High|
|

In [0]:
driver_safety_score = spark.table("gold_driver_safety_score")

In [0]:
from pyspark.sql.functions import avg, col, when

# ==========================================
# GOLD STEP 6 - DRIVER LEVEL SUMMARY
# ==========================================

# Aggregate to one row per driver
driver_summary = driver_safety_score.groupBy("drivername", "VehicleNo") \
    .agg(avg("safety_score").alias("avg_safety_score"))

# Assign final risk rating based on average score
driver_summary = driver_summary.withColumn(
    "final_risk_rating",
    when(col("avg_safety_score") >= 80, "Low")
    .when((col("avg_safety_score") >= 50) & (col("avg_safety_score") < 80), "Medium")
    .otherwise("High")
)

# Save as new Gold table
driver_summary.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_driver_summary")

# Preview
driver_summary.show(10)

+----------+------------+------------------+-----------------+
|drivername|   VehicleNo|  avg_safety_score|final_risk_rating|
+----------+------------+------------------+-----------------+
|  driver32|MSNP00032324| 88.29999999999998|              Low|
|   driver4|  TG16UL5678| 45.04615384615385|             High|
|  driver31| MSNLG903136| 87.52307692307691|              Low|
|  driver33|  TS07UK3537| 39.62307692307694|             High|
|   driver3|   TS19T7317|14.646153846153847|             High|
|  driver23|MSNHYND10032| 86.26923076923077|              Low|
|  driver20|  TS04UC8236| 61.30769230769231|           Medium|
|  driver29|    MSN29306| 51.25384615384615|           Medium|
|   driver6|  TS29TA4254| 49.34615384615385|             High|
|  driver21|  TS02UD0159|  89.0153846153846|              Low|
+----------+------------+------------------+-----------------+
only showing top 10 rows
